<a href="https://colab.research.google.com/github/vaishnavi2810-code/AI-For-Beginners/blob/main/shakespeare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interpreting Classifier Weights

In this experiment, you will train models to distringuish examples of two different genres of Shakespeare's plays: comedies and tragedies. (We'll ignore the histories, sonnets, etc.) Since he died four hundred years ago, Shakespeare has not written any more plays—although scraps of various other works have come to light. We are not, therefore, interested in building models simply to help categorize an unbounded stream of future documents, as we might be in other applications of text classification; rather, we are interested in what a classifier might have to tell us about what we mean by the terms “comedy” and “tragedy”.

You will start by copying and running your `createBasicFeatures` function from the experiment with movie reviews. Do the features the classifier focuses on tell you much about comedy and tragedy in general?

You will then implement another featurization function `createInterestingFeatures`, which will focus on only those features you think are informative for distinguishing between comedy and tragedy. Accuracy on leave-one-out cross-validation may go up, but it more important to look at the features given the highest weight by the classifier. Interpretability in machine learning, of course, may be harder to define than accuracy—although accuracy at some tasks is hard enoough.

In [6]:
import json
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate,LeaveOneOut
import numpy as np

In [7]:
#read in the shakespeare corpus
def readShakespeare():
  raw = requests.get("https://raw.githubusercontent.com/dasmiq/cs6120-assignment2/refs/heads/main/shakespeare_plays.json").text.strip()
  corpus = [json.loads(line) for line in raw.split("\n")]

  #remove histories from the data, as we're only working with tragedies and comedies
  corpus = [entry for entry in corpus if entry["genre"] != "history"]
  return corpus

This is where you will implement two functions to featurize the data:

In [4]:
# TODO: Implement createBasicFeatures
# NB: The current contents are for testing only
# This function should return:
#  -a sparse numpy matrix of document features
#  -a list of the correct genre for each document
#  -a list of the vocabulary used by the features, such that the ith term of the
#    list is the word whose counts appear in the ith column of the matrix.

# This function should create a feature representation using all tokens that
# contain an alphabetic character.
from sklearn.feature_extraction.text import CountVectorizer
import re
def createBasicFeatures(corpus):
  #Your code here
  texts = [doc["text"] for doc in corpus]
  classes = [doc["genre"] for doc in corpus]

  # Define token pattern: keep tokens with at least one alphabetic character
  token_pattern = r"(?u)\b\w*[A-Za-z]\w*\b"

  # Use CountVectorizer to build the matrix
  vectorizer = CountVectorizer(token_pattern=token_pattern, lowercase=True)
  X = vectorizer.fit_transform(texts)

  # Get vocab list in the correct column order. It tells us which word is there corresponding to each column.
  vocab = vectorizer.get_feature_names_out().tolist()
  return X,classes,vocab

In [15]:
# TODO: Implement createInterestingFeatures. Describe your features and what
# they might tell you about the difference between comedy and tragedy.
# This function can add other features you want that help classification
# accuracy, such as bigrams, word prefixes and suffixes, etc.
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
def createInterestingFeatures(corpus):
  #Your code here
  """
  Features for interpreting comedy vs tragedy:
  1. TF-IDF of unigrams + bigrams (captures key words/phrases)
  2. Document length (optional: can help with verbosity differences)
  """
  texts = [doc["text"] for doc in corpus]
  genres = [doc["genre"] for doc in corpus]

  # --- TF-IDF unigrams + bigrams ---
  token_pattern = r"(?u)\b\w*[A-Za-z]\w*\b"
  vectorizer = TfidfVectorizer(token_pattern=token_pattern,
                                lowercase=True,
                                ngram_range=(1,3),  # unigrams + bigrams
                                min_df=2)
  X_tfidf = vectorizer.fit_transform(texts)

  # --- Optional numeric features ---
  doc_lengths = np.array([len(t.split()) for t in texts]).reshape(-1, 1)

  # Combine into single feature matrix
  X = hstack([X_tfidf, doc_lengths])

  # Build vocabulary list
  vocab = vectorizer.get_feature_names_out().tolist()
  vocab += ["doc_length"]

  return X, genres, vocab

In [13]:
#given a numpy matrix representation of the features for the training set, the
# vector of true classes for each example, and the vocabulary as described
# above, this computes the accuracy of the model using leave one out cross
# validation and reports the most indicative features for each class
def evaluateModel(X,y,vocab,penalty="l1"):
  #create and fit the model
  model = LogisticRegression(penalty=penalty,solver="liblinear")
  results = cross_validate(model,X,y,cv=LeaveOneOut())

  #determine the average accuracy
  scores = results["test_score"]
  avg_score = sum(scores)/len(scores)

  #determine the most informative features
  # this requires us to fit the model to everything, because we need a
  # single model to draw coefficients from, rather than 26
  model.fit(X,y)
  neg_class_prob_sorted = model.coef_[0, :].argsort()
  pos_class_prob_sorted = (-model.coef_[0, :]).argsort()

  termsToTake = 20
  pos_indicators = [vocab[i] for i in neg_class_prob_sorted[:termsToTake]]
  neg_indicators = [vocab[i] for i in pos_class_prob_sorted[:termsToTake]]

  return avg_score,pos_indicators,neg_indicators

def runEvaluation(X,y,vocab):
  print("----------L1 Norm-----------")
  avg_score,pos_indicators,neg_indicators = evaluateModel(X,y,vocab,"l1")
  print("The model's average accuracy is %f"%avg_score)
  print("The most informative terms for pos are: %s"%pos_indicators)
  print("The most informative terms for neg are: %s"%neg_indicators)
  #this call will fit a model with L2 normalization
  print("----------L2 Norm-----------")
  avg_score,pos_indicators,neg_indicators = evaluateModel(X,y,vocab,"l2")
  print("The model's average accuracy is %f"%avg_score)
  print("The most informative terms for pos are: %s"%pos_indicators)
  print("The most informative terms for neg are: %s"%neg_indicators)


In [8]:
corpus = readShakespeare()
print(corpus)

[{'genre': 'comedy', 'id': 'allsWellThatEndsWell', 'text': 'alls well that ends well by william shakespeare dramatis personae king of france the duke of florence bertram count of rousillon lafeu an old lord parolles a follower of bertram two french lords serving with bertram steward servant to the countess of rousillon lavache a clown and servant to the countess of rousillon a page servant to the countess of rousillon countess of rousillon mother to bertram helena a gentlewoman protected by the countess a widow of florence diana daughter to the widow violenta neighbour and friend to the widow mariana neighbour and friend to the widow lords officers soldiers etc french and florentine scene rousillon paris florence marseilles act i scene 1 rousillon the count s palace enter bertram the countess of rousillon helena and lafeu all in black countess in delivering my son from me i bury a second husband bertram and i in going madam weep o er my father s death anew but i must attend his majesty

Run the following to train and evaluate two models with basic features:

In [11]:
X,y,vocab = createBasicFeatures(corpus)
runEvaluation(X, y, vocab)

----------L1 Norm-----------
The model's average accuracy is 0.615385
The most informative terms for pos are: ['you', 'helena', 'duke', 'prospero', 'i', 'sir', 'leontes', 'a', 'of', 'presently', 'preservers', 'preserver', 'preserved', 'pretty', 'prettiness', 'prettily', 'prettiest', 'prettier', 'presentment', 'preserving']
The most informative terms for neg are: ['him', 's', 'iago', 'imogen', 'o', 'brutus', 'lear', 'ham', 'and', 'rom', 'the', 'presentation', 'prettily', 'prettiest', 'prettier', 'pretia', 'pretext', 'pretense', 'pretending', 'presenters']
----------L2 Norm-----------
The model's average accuracy is 0.769231
The most informative terms for pos are: ['i', 'you', 'duke', 'prospero', 'a', 'helena', 'your', 'antonio', 'sir', 'leontes', 'hermia', 'for', 'lysander', 'ariel', 'sebastian', 'demetrius', 'camillo', 'stephano', 'me', 'parolles']
The most informative terms for neg are: ['iago', 'othello', 's', 'him', 'imogen', 'what', 'lear', 'brutus', 'his', 'cassio', 'o', 'ham', 'o

Run the following to train and evaluate two models with features that are interesting for distinguishing comedy and tragedy:

In [16]:
X,y,vocab = createInterestingFeatures(corpus)
runEvaluation(X, y, vocab)

----------L1 Norm-----------
The model's average accuracy is 0.000000
The most informative terms for pos are: ['s soul', 's steward', 's statue which', 's statue', 's state', 's staff', 's spring', 's spoken', 's spirit', 's spent', 's speed', 's sound', 's storm', 's sorrow', 's sons', 's song come', 's song', 's son who', 's son his', 's son and']
The most informative terms for neg are: ['doc_length', 'youth with', 'youth will', 'youth when', 'youth to', 'youth thou', 'youth there is', 'youth there', 'youth the', 'youth that he', 'youth that', 'youth s', 'youth of the', 'youth of', 'youth my', 'youth let', 'youth is', 'a band of', 'a band', 'a ballad']
----------L2 Norm-----------
The model's average accuracy is 0.615385
The most informative terms for pos are: ['i', 'you', 'a', 'of', 'sir', 'me', 'duke', 'ford', 'for', 'is', 'your', 'her', 'my', 'to', 'petruchio', 'will', 'angelo', 'it', 'mrs', 'helena']
The most informative terms for neg are: ['timon', 'antony', 'brutus', 'caesar', 

**TODO**: Based on the most informative features in the output of the classifier evaluation, what do these classifiers tell you about the differences between comedy and tragedy?